# CausalTimePrior: Algorithm Walkthrough

**Interventional Time Series Priors for Causal Foundation Models**

This notebook provides a step-by-step walkthrough of the CausalTimePrior data generation algorithm. We trace through the actual source code to explain:

1. **How does the algorithm sample?** — Full generation pipeline
2. **How many lags?** — Temporal graph structure
3. **Erdős-Rényi model & priors** — Graph, mechanism, and noise priors
4. **Beta prior for edge probability** — Sparse graph generation
5. **Intervention types** — Hard, soft, and time-varying interventions
6. **How many interventions?** — Targets, timing, and structure
7. **Regime-switching** — Markov-driven structural breaks

In [1]:
import sys
import torch
import torch.nn as nn
import torch.distributions as dist
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from causal_time_prior import (
    CausalTimePrior, TemporalSCM, TemporalDAG, TemporalGraphBuilder,
    TemporalMechanism, TemporalSCMBuilder, InterventionSampler,
    InterventionSpec, InterventionType, DEFAULT_CONFIG
)
from causal_time_prior.chain_scm import ChainSCMBuilder
from causal_time_prior.regime_switching_builder import RegimeSwitchingSCMBuilder
from causal_time_prior.regime_switching import RegimeSwitchingTemporalSCM
from dopfnprior.utils.sampling import ShiftedExponentialSampler

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Default config: {DEFAULT_CONFIG}")

PyTorch version: 2.3.1+rocm5.7
Default config: {'N_max': 10, 'K_max': 3, 'alpha': 2, 'beta': 5, 'gamma': 0.7, 'sigma_w': 1.0, 'sigma_b': 0.5, 'T': 100, 'burn_in': 50, 'root_mean': 0.0, 'non_root_mean': 0.0, 'device': device(type='cpu'), 'dtype': torch.float32}


---
## 1. How Does the Algorithm Sample?

The entry point is `CausalTimePrior.generate_pair(T)`, which produces a paired observational and interventional time series from a single SCM.

### Call chain

```
generate_pair(T=100)
  ├── sample_scm()                          # Step 1: Sample SCM structure
  │   ├── 70%: Diverse nonlinear SCM         (TemporalSCMBuilder)
  │   ├── 15%: Chain SCM (A→B→C→...)         (ChainSCMBuilder)
  │   └── 15%: Regime-switching SCM           (RegimeSwitchingSCMBuilder)
  │
  ├── InterventionSampler(N, T).sample()     # Step 2: Sample intervention
  │   ├── Type: 50% hard, 30% soft, 20% time-varying
  │   ├── Targets: 1–2 variables
  │   └── Times: contiguous window [start, start+length)
  │
  ├── scm.sample_observational(T, burn_in)   # Step 3: Forward simulate (no intervention)
  │
  └── scm.sample_interventional(T, intv)     # Step 4: Forward simulate (with intervention)
```

### SCM type routing

A uniform random draw determines the SCM type (see `prior.py:121–177`):

In [2]:
# Demonstrate SCM type routing
prior = CausalTimePrior(seed=42)

print(f"Chain probability:           {prior.chain_prob:.0%}")
print(f"Regime-switching probability: {prior.regime_switching_prob:.0%}")
print(f"Diverse nonlinear:           {1 - prior.chain_prob - prior.regime_switching_prob:.0%}")
print()

# Sample 200 SCMs and count types
prior_count = CausalTimePrior(seed=123)
type_counts = {'diverse': 0, 'chain': 0, 'regime_switching': 0}
for _ in range(200):
    rand_val = torch.rand(1, generator=prior_count.generator).item()
    if rand_val < prior_count.chain_prob:
        type_counts['chain'] += 1
    elif rand_val < prior_count.chain_prob + prior_count.regime_switching_prob:
        type_counts['regime_switching'] += 1
    else:
        type_counts['diverse'] += 1

print("Empirical SCM type distribution (200 samples):")
for k, v in type_counts.items():
    print(f"  {k:20s}: {v:3d} ({v/200:.1%})")

Chain probability:           15%
Regime-switching probability: 15%
Diverse nonlinear:           70%

Empirical SCM type distribution (200 samples):
  diverse             : 143 (71.5%)
  chain               :  31 (15.5%)
  regime_switching    :  26 (13.0%)


In [3]:
# Generate a single paired example
prior = CausalTimePrior(seed=42)
X_obs, X_int, intervention, scm = prior.generate_pair(T=100)

N = X_obs.shape[1]
print(f"Number of variables (N):     {N}")
print(f"Time series length (T):      {X_obs.shape[0]}")
print(f"Number of lags (K):          {scm.dag.K}")
print(f"Observational shape:         {X_obs.shape}")
print(f"Interventional shape:        {X_int.shape}")
print(f"Intervention type:           {intervention.intervention_type.value}")
print(f"Intervention targets:        {intervention.targets}")
print(f"Intervention time window:    [{intervention.times[0]}, {intervention.times[-1]}]")

Number of variables (N):     6
Time series length (T):      100
Number of lags (K):          2
Observational shape:         torch.Size([100, 6])
Interventional shape:        torch.Size([100, 6])
Intervention type:           hard
Intervention targets:        [3, 0]
Intervention time window:    [17, 90]


### Building a single SCM — step by step

For the **diverse nonlinear** path (70% of samples), the builder performs these steps:

1. **Sample hyperparameters**: N ~ Uniform[3, 10], K ~ Uniform[1, 3], p ~ Beta(2, 5)
2. **Sample graph**: G_0 (acyclic, via Do-PFN's ER model) + G_1...G_K (lagged Bernoulli matrices)
3. **Sample mechanisms**: For each node, sample activation function and weights
4. **Sample noise**: Root nodes get larger noise; non-root nodes get one of 3 noise families

In [4]:
# Step-by-step SCM construction
gen = torch.Generator().manual_seed(42)

# Step 1: Sample hyperparameters
N = int(torch.randint(3, 11, (1,), generator=gen).item())    # Uniform[3, 10]
K = int(torch.randint(1, 4, (1,), generator=gen).item())     # Uniform[1, 3]
p = float(dist.Beta(2, 5).sample().item())                   # Beta(2, 5)
dropout = float(torch.rand(1, generator=gen).item() * 0.3)   # Uniform[0, 0.3]

print("Step 1 — Sampled hyperparameters:")
print(f"  N (variables):       {N}")
print(f"  K (max lag):         {K}")
print(f"  p (edge prob):       {p:.3f}")
print(f"  dropout:             {dropout:.3f}")
print()

# Step 2: Sample temporal DAG
graph_builder = TemporalGraphBuilder(
    num_nodes=N, edge_prob=p, dropout_prob=dropout, max_lag=K, gamma=0.7
)
dag = graph_builder.sample(gen)

print("Step 2 — Sampled temporal DAG:")
print(f"  G_0 nodes:           {list(dag.G_0.nodes())}")
print(f"  G_0 edges:           {list(dag.G_0.edges())}")
print(f"  Topological order:   {dag.topo_order}")
for k in range(K):
    nnz = int(dag.G_lags[k].sum())
    total = dag.G_lags[k].size
    print(f"  G_{k+1} non-zero:     {nnz}/{total} (p_{k+1} = {p * 0.7**(k+1):.3f})")

Step 1 — Sampled hyperparameters:
  N (variables):       9
  K (max lag):         3
  p (edge prob):       0.122
  dropout:             0.115

Step 2 — Sampled temporal DAG:
  G_0 nodes:           ['x0', 'x1', 'x2', 'x3', 'u4', 'u5', 'x6', 'x7', 'y']
  G_0 edges:           [('x3', 'u4'), ('u5', 'u4'), ('x6', 'u4'), ('x7', 'y'), ('y', 'x2')]
  Topological order:   ['x0', 'x1', 'x3', 'u5', 'x6', 'x7', 'u4', 'y', 'x2']
  G_1 non-zero:     7/81 (p_1 = 0.086)
  G_2 non-zero:     6/81 (p_2 = 0.060)
  G_3 non-zero:     7/81 (p_3 = 0.042)


### Forward simulation

The time-stepped simulation in `temporal_scm.py` processes each timestep sequentially. At each step t, nodes are evaluated in **topological order** of G_0:

```
for t in range(T + burn_in):
    for v in topological_order(G_0):
        # Gather instantaneous parents from G_0
        Pa_instant = {u: buffer[t, u] for u in parents(v, G_0)}
        
        # Gather lagged parents from G_1, ..., G_K
        Pa_lagged = [{u: buffer[t-k, u] for u in parents(v, G_k)} for k in 1..K]
        
        # Evaluate mechanism
        buffer[t, v] = activation(W_inst · Pa_instant + W_lag · Pa_lagged + bias) + noise
```

Key properties:
- **Burn-in** (default 50 steps): discarded to reach stationary distribution
- **Clipping**: values clamped to [-1000, 1000] at each step
- **Divergence check**: after simulation, rejects if any |x| > 10^6 or NaN/Inf

---
## 2. How Many Lags Do the Temporal Graphs Have?

The maximum lag **K** is sampled per SCM:

$$K \sim \text{Uniform}\{1, 2, 3\}$$

This means each temporal SCM has **1 to 3 lag matrices** (G_1, ..., G_K). Lagged edges capture dependencies like "X_i at time t depends on X_j at time t-k".

The edge probability for lag k decays geometrically:

$$p_k = p \cdot \gamma^k, \quad \gamma = 0.7$$

So recent lags have denser connectivity than distant lags. Lagged edges have **no acyclicity constraint** (unlike G_0), because temporal ordering already prevents causal cycles.

In [5]:
# Demonstrate lag structure
gamma = 0.7
p_base = 0.3  # Example edge probability

print("Lag decay example (p = 0.3, gamma = 0.7):")
print(f"  G_0 (instantaneous): p_0 = {p_base:.3f}")
for k in range(1, 4):
    p_k = p_base * gamma**k
    print(f"  G_{k} (lag {k}):          p_{k} = {p_k:.3f}")

print()

# Empirical K distribution
gen = torch.Generator().manual_seed(0)
K_samples = [int(torch.randint(1, 4, (1,), generator=gen).item()) for _ in range(1000)]
print(f"Empirical K distribution (1000 samples):")
for k in [1, 2, 3]:
    count = K_samples.count(k)
    print(f"  K={k}: {count} ({count/10:.1f}%)")

# Special case: Chain SCMs always use K=2
print(f"\nChain SCMs: K = 2 (fixed, see chain_scm.py:76)")

Lag decay example (p = 0.3, gamma = 0.7):
  G_0 (instantaneous): p_0 = 0.300
  G_1 (lag 1):          p_1 = 0.210
  G_2 (lag 2):          p_2 = 0.147
  G_3 (lag 3):          p_3 = 0.103

Empirical K distribution (1000 samples):
  K=1: 322 (32.2%)
  K=2: 345 (34.5%)
  K=3: 333 (33.3%)

Chain SCMs: K = 2 (fixed, see chain_scm.py:76)


In [6]:
# Visualize temporal DAG structure: G_0, G_1, ..., G_K side by side
gen = torch.Generator().manual_seed(15)
N_vis, K_vis, p_vis = 5, 3, 0.30

graph_builder = TemporalGraphBuilder(
    num_nodes=N_vis, edge_prob=p_vis, dropout_prob=0.0, max_lag=K_vis, gamma=0.7
)
dag_vis = graph_builder.sample(gen)

# Use consistent node positions across all panels
pos = nx.shell_layout(dag_vis.G_0)

fig, axes = plt.subplots(1, K_vis + 1, figsize=(4.5 * (K_vis + 1), 4))

# Panel 0: G_0 (instantaneous, acyclic)
ax = axes[0]
nx.draw(dag_vis.G_0, pos, ax=ax, with_labels=True, node_color='#AED6F1',
        node_size=600, font_size=10, font_weight='bold',
        edge_color='steelblue', width=2.0, arrows=True, arrowsize=20,
        connectionstyle='arc3,rad=0.1')
n_edges_0 = dag_vis.G_0.number_of_edges()
ax.set_title(f'$G_0$ (instantaneous)\n{n_edges_0} edges, acyclic', fontsize=11)

# Panels 1..K: Lagged graphs G_k
for k in range(K_vis):
    ax = axes[k + 1]
    G_k_mat = dag_vis.G_lags[k]
    
    # Convert adjacency matrix to DiGraph with the same node names
    G_k_graph = nx.DiGraph()
    G_k_graph.add_nodes_from(dag_vis.topo_order)
    for i in range(N_vis):
        for j in range(N_vis):
            if G_k_mat[i, j] > 0:
                G_k_graph.add_edge(dag_vis.topo_order[i], dag_vis.topo_order[j])
    
    nx.draw(G_k_graph, pos, ax=ax, with_labels=True, node_color='#FADBD8',
            node_size=600, font_size=10, font_weight='bold',
            edge_color='coral', width=2.0, arrows=True, arrowsize=20,
            connectionstyle='arc3,rad=0.1')
    n_edges_k = int(G_k_mat.sum())
    p_k = p_vis * 0.7**(k+1)
    ax.set_title(f'$G_{{{k+1}}}$ (lag {k+1})\n{n_edges_k} edges, $p_{{{k+1}}}$={p_k:.3f}', fontsize=11)

plt.suptitle(f'Temporal DAG structure (N={N_vis}, K={K_vis}, p={p_vis})\n'
             r'Edge labels: $X_i \to X_j$ in $G_0$ means $X_i(t) \to X_j(t)$; '
             r'in $G_k$ means $X_i(t\!-\!k) \to X_j(t)$',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

/tmp/ipykernel_12977/3165588503.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Erdős-Rényi Graph Model and Priors

### Graph prior

The instantaneous graph G_0 is sampled via Do-PFN's `GraphBuilder`, which uses the **Erdős-Rényi (ER) model**:
- Each directed edge (i, j) exists independently with probability p
- **Acyclicity is enforced** by restricting edges to follow a sampled topological order
- The maximum number of possible edges in a DAG on N nodes is N(N-1)/2

Lagged graphs G_1, ..., G_K are full N x N Bernoulli matrices (any variable can depend on any other variable's past values — no acyclicity needed).

In [7]:
# Sample and visualize a temporal DAG
gen = torch.Generator().manual_seed(10)
N_ex, K_ex, p_ex = 5, 2, 0.35

graph_builder = TemporalGraphBuilder(
    num_nodes=N_ex, edge_prob=p_ex, dropout_prob=0.0, max_lag=K_ex, gamma=0.7
)
dag = graph_builder.sample(gen)

print(f"Temporal DAG: N={N_ex}, K={K_ex}, p={p_ex}")
print(f"Topo order: {dag.topo_order}")
print()

# G_0: instantaneous edges (DAG)
print("G_0 (instantaneous, acyclic):")
for u, v in dag.G_0.edges():
    print(f"  {u} -> {v}")
print(f"  Total edges: {dag.G_0.number_of_edges()} / {N_ex*(N_ex-1)//2} possible")
print()

# G_1, G_2: lagged edges
for k in range(K_ex):
    G_k = dag.G_lags[k]
    nnz = int(G_k.sum())
    print(f"G_{k+1} (lag {k+1}, p_{k+1}={p_ex * 0.7**(k+1):.3f}):")
    print(f"  Adjacency matrix:\n{G_k.astype(int)}")
    for i in range(N_ex):
        for j in range(N_ex):
            if G_k[i, j] > 0:
                print(f"  {dag.topo_order[i]}(t-{k+1}) -> {dag.topo_order[j]}(t)")
    print(f"  Total edges: {nnz} / {N_ex*N_ex} possible")
    print()

Temporal DAG: N=5, K=2, p=0.35
Topo order: ['x0', 'x1', 'x2', 'y', 'x3']

G_0 (instantaneous, acyclic):
  x0 -> x2
  x0 -> y
  y -> x3
  Total edges: 3 / 10 possible

G_1 (lag 1, p_1=0.245):
  Adjacency matrix:
[[1 1 0 0 0]
 [0 1 0 1 0]
 [1 0 0 0 1]
 [0 1 0 0 1]
 [1 0 1 1 0]]
  x0(t-1) -> x0(t)
  x0(t-1) -> x1(t)
  x1(t-1) -> x1(t)
  x1(t-1) -> y(t)
  x2(t-1) -> x0(t)
  x2(t-1) -> x3(t)
  y(t-1) -> x1(t)
  y(t-1) -> x3(t)
  x3(t-1) -> x0(t)
  x3(t-1) -> x2(t)
  x3(t-1) -> y(t)
  Total edges: 11 / 25 possible

G_2 (lag 2, p_2=0.171):
  Adjacency matrix:
[[0 0 0 1 0]
 [0 1 0 0 0]
 [0 0 0 0 0]
 [1 1 0 0 0]
 [0 0 0 0 0]]
  x0(t-2) -> y(t)
  x1(t-2) -> x1(t)
  y(t-2) -> x0(t)
  y(t-2) -> x1(t)
  Total edges: 4 / 25 possible



In [8]:
# Visualize the sampled temporal DAG from above
pos = nx.shell_layout(dag.G_0)

fig, axes = plt.subplots(1, K_ex + 1, figsize=(4.5 * (K_ex + 1), 4))

# G_0
ax = axes[0]
nx.draw(dag.G_0, pos, ax=ax, with_labels=True, node_color='#AED6F1',
        node_size=600, font_size=10, font_weight='bold',
        edge_color='steelblue', width=2.0, arrows=True, arrowsize=20,
        connectionstyle='arc3,rad=0.1')
ax.set_title(f'$G_0$ (instantaneous, acyclic)\n{dag.G_0.number_of_edges()} edges', fontsize=11)

# G_1 ... G_K
for k in range(K_ex):
    ax = axes[k + 1]
    G_k_mat = dag.G_lags[k]
    
    G_k_graph = nx.DiGraph()
    G_k_graph.add_nodes_from(dag.topo_order)
    for i in range(N_ex):
        for j in range(N_ex):
            if G_k_mat[i, j] > 0:
                G_k_graph.add_edge(dag.topo_order[i], dag.topo_order[j])
    
    nx.draw(G_k_graph, pos, ax=ax, with_labels=True, node_color='#FADBD8',
            node_size=600, font_size=10, font_weight='bold',
            edge_color='coral', width=2.0, arrows=True, arrowsize=20,
            connectionstyle='arc3,rad=0.1')
    n_edges = int(G_k_mat.sum())
    p_k = p_ex * 0.7**(k+1)
    ax.set_title(f'$G_{{{k+1}}}$ (lag {k+1})\n{n_edges} edges, $p_{{{k+1}}}$={p_k:.3f}', fontsize=11)

plt.suptitle(f'Erdős-Rényi temporal DAG (N={N_ex}, K={K_ex}, p={p_ex})', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

/tmp/ipykernel_12977/4119998900.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Mechanism prior

Each node's structural equation is:

$$X_t^{(i)} = \sigma\!\left(\sum_{j \in \text{Pa}_{G_0}(i)} w_j^{\text{inst}} \cdot X_t^{(j)} + \sum_{k=1}^{K} \sum_{j \in \text{Pa}_{G_k}(i)} w_{j,k}^{\text{lag}} \cdot X_{t-k}^{(j)} + b\right) + \varepsilon_t^{(i)}$$

where:
- **Activation** $\sigma$ is sampled uniformly from **9 functions**: Identity, Tanh, TanhX2, TanhReLU, ReLU, Sin, Cos, Abs, Square
- **Weights** $w \sim \mathcal{N}(0, \sigma_w^2)$ with $\sigma_w = 1.0$
- **Bias** $b \sim \mathcal{N}(0, \sigma_b^2)$ with $\sigma_b = 0.5$

In [9]:
# List all 9 activation functions
prior = CausalTimePrior(seed=0)

print("9 activation functions (sampled uniformly per node):")
activation_names = [
    "Identity (linear)", "Tanh", "TanhX2 = tanh(x^2)", "TanhReLU = tanh(relu(x))",
    "ReLU", "Sin", "Cos", "Abs", "Square = x^2"
]
for i, (act, name) in enumerate(zip(prior.activations, activation_names)):
    print(f"  [{i}] {name:30s}  ({type(act).__name__})")

print(f"\nWeight prior:  w ~ N(0, {DEFAULT_CONFIG['sigma_w']}^2)")
print(f"Bias prior:    b ~ N(0, {DEFAULT_CONFIG['sigma_b']}^2)")

# Visualize activation functions
x = torch.linspace(-3, 3, 200)
fig, axes = plt.subplots(1, 9, figsize=(18, 2))
for i, (act, name) in enumerate(zip(prior.activations, activation_names)):
    y = act(x)
    axes[i].plot(x.numpy(), y.detach().numpy(), 'b-', linewidth=1.5)
    axes[i].set_title(name.split('=')[0].strip(), fontsize=8)
    axes[i].axhline(0, color='gray', linewidth=0.5)
    axes[i].axvline(0, color='gray', linewidth=0.5)
    axes[i].set_xlim(-3, 3)
    axes[i].tick_params(labelsize=6)
plt.suptitle('Mechanism activation functions', fontsize=11)
plt.tight_layout()
plt.show()

9 activation functions (sampled uniformly per node):
  [0] Identity (linear)               (Identity)
  [1] Tanh                            (Tanh)
  [2] TanhX2 = tanh(x^2)              (TanhX2)
  [3] TanhReLU = tanh(relu(x))        (TanhReLU)
  [4] ReLU                            (ReLU)
  [5] Sin                             (Sin)
  [6] Cos                             (Cos)
  [7] Abs                             (Abs)
  [8] Square = x^2                    (Square)

Weight prior:  w ~ N(0, 1.0^2)
Bias prior:    b ~ N(0, 0.5^2)


/tmp/ipykernel_12977/4075777327.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Noise prior

Noise distributions differ for root and non-root nodes:

| Node type | Std sampling | Distribution families |
|-----------|-------------|----------------------|
| **Root** (no parents in G_0) | std ~ ShiftedExp(rate=1.0, shift=0.1) | Gaussian only |
| **Non-root** (has parents) | std ~ ShiftedExp(rate=10.0, shift=0.01) | 1/3 Gaussian, 1/3 Uniform, 1/3 Laplace |

The ShiftedExponential samples as: $\text{std} = \text{Exp}(\text{rate}) + \text{shift}$.

Root nodes have larger noise (they generate the "driving signal"), while non-root nodes have smaller noise (most of their variance comes from parent mechanisms).

In [10]:
# Demonstrate noise prior sampling
gen = torch.Generator().manual_seed(42)

# Sample root noise stds
root_dist = ShiftedExponentialSampler(rate=1.0, shift=0.1)
root_stds = [root_dist.sample(generator=gen) for _ in range(1000)]

# Sample non-root noise stds
nonroot_dist = ShiftedExponentialSampler(rate=10.0, shift=0.01)
nonroot_stds = [nonroot_dist.sample(generator=gen) for _ in range(1000)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))

ax1.hist(root_stds, bins=50, density=True, alpha=0.7, color='steelblue')
ax1.set_title('Root node noise std\nShiftedExp(rate=1.0, shift=0.1)')
ax1.set_xlabel('std')
ax1.axvline(np.mean(root_stds), color='red', linestyle='--', label=f'mean={np.mean(root_stds):.2f}')
ax1.legend(fontsize=8)

ax2.hist(nonroot_stds, bins=50, density=True, alpha=0.7, color='coral')
ax2.set_title('Non-root node noise std\nShiftedExp(rate=10.0, shift=0.01)')
ax2.set_xlabel('std')
ax2.axvline(np.mean(nonroot_stds), color='red', linestyle='--', label=f'mean={np.mean(nonroot_stds):.2f}')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Non-root noise family (each with 1/3 probability):")
print("  Gaussian:  N(0, std)")
print("  Uniform:   U(-a, a)   where a = std * sqrt(3)  [variance-matched]")
print("  Laplace:   Lap(0, b)  where b = std / sqrt(2)  [variance-matched]")

Non-root noise family (each with 1/3 probability):
  Gaussian:  N(0, std)
  Uniform:   U(-a, a)   where a = std * sqrt(3)  [variance-matched]
  Laplace:   Lap(0, b)  where b = std / sqrt(2)  [variance-matched]


/tmp/ipykernel_12977/2501932551.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 4. Beta Prior for Edge Probability

The edge probability p for each SCM is sampled from a **Beta distribution**:

$$p \sim \text{Beta}(\alpha=2, \beta=5)$$

Properties:
- **Mean** = α/(α+β) = 2/7 ≈ 0.286
- **Mode** = (α-1)/(α+β-2) = 1/5 = 0.200
- **Variance** = αβ / ((α+β)²(α+β+1)) ≈ 0.026

This biases the prior toward **sparse graphs**, matching the typical sparsity in real-world causal systems.

In [11]:
# Plot Beta(2,5) distribution and lag decay
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Beta(2,5) PDF
x = torch.linspace(0.001, 0.999, 500)
beta_dist = dist.Beta(2, 5)
pdf = torch.exp(beta_dist.log_prob(x))

ax1.plot(x.numpy(), pdf.numpy(), 'b-', linewidth=2)
ax1.fill_between(x.numpy(), pdf.numpy(), alpha=0.2)
ax1.axvline(2/7, color='red', linestyle='--', linewidth=1.5, label=f'Mean = 2/7 = {2/7:.3f}')
ax1.axvline(1/5, color='orange', linestyle='--', linewidth=1.5, label=f'Mode = 1/5 = {1/5:.3f}')
ax1.set_xlabel('Edge probability p')
ax1.set_ylabel('Density')
ax1.set_title('Beta(2, 5) prior over edge probability')
ax1.legend()

# Lag decay
gamma = 0.7
p_values = [0.15, 0.25, 0.35, 0.45]
lags = np.arange(0, 4)

for p in p_values:
    p_k = [p * gamma**k for k in lags]
    ax2.plot(lags, p_k, 'o-', linewidth=2, markersize=8, label=f'p = {p:.2f}')

ax2.set_xlabel('Lag k')
ax2.set_ylabel('Edge probability p_k')
ax2.set_title(r'Lag decay: $p_k = p \cdot \gamma^k$, $\gamma=0.7$')
ax2.set_xticks([0, 1, 2, 3])
ax2.set_xticklabels(['G_0', 'G_1', 'G_2', 'G_3'])
ax2.legend()
ax2.set_ylim(0, 0.5)

plt.tight_layout()
plt.show()

# Concrete example
p = 0.286  # mean of Beta(2,5)
N = 5
print(f"\nConcrete example: N={N}, p={p:.3f}, gamma=0.7")
print(f"  G_0: max possible edges = {N*(N-1)//2}, expected = {N*(N-1)//2 * p:.1f}")
for k in range(1, 4):
    p_k = p * gamma**k
    print(f"  G_{k}: max possible edges = {N*N}, expected = {N*N * p_k:.1f} (p_{k} = {p_k:.3f})")


Concrete example: N=5, p=0.286, gamma=0.7
  G_0: max possible edges = 10, expected = 2.9
  G_1: max possible edges = 25, expected = 5.0 (p_1 = 0.200)
  G_2: max possible edges = 25, expected = 3.5 (p_2 = 0.140)
  G_3: max possible edges = 25, expected = 2.5 (p_3 = 0.098)


/tmp/ipykernel_12977/1578048594.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 5. Intervention Types

CausalTimePrior supports three intervention types, sampled with probabilities:

| Type | Probability | Mathematical form | Effect |
|------|------------|-------------------|--------|
| **Hard** | 50% | $\text{do}(X_i := c)$ | Replaces mechanism entirely with constant c |
| **Soft** | 30% | $X_i = f_i(\text{Pa}) + \varepsilon + \delta$ | Adds shift $\delta$ to mechanism output |
| **Time-varying** | 20% | $\text{do}(X_i := c(t))$ | Replaces with time-dependent function c(t) |

### Hard intervention: do(X_i := c)
- Value: $c \sim \mathcal{N}(0, 4)$ (i.e., `randn() * 2.0`)
- During active time window: the variable is **set to c**, ignoring its parents entirely
- This is the classic Pearl do-operator

### Soft intervention: mechanism perturbation
- Shift: $\delta \sim \mathcal{N}(0, 1)$
- During active time window: the mechanism still runs, but $\delta$ is **added** to the output
- Parents still influence the variable; the intervention only shifts the distribution

### Time-varying intervention: do(X_i := c(t))
Four profile types, each sampled with equal probability:
1. **Step**: jumps from -2 to +2 at midpoint
2. **Ramp**: linearly interpolates from -2 to +2
3. **Sinusoidal**: $c(t) = 2 \sin(2\pi t / L)$
4. **Trajectory**: independent $c(t) \sim \mathcal{N}(0, 4)$ at each timestep

In [12]:
# Demonstrate all intervention types
N_demo, T_demo = 5, 100

# Sample one of each type by controlling the seed
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

# --- HARD intervention ---
gen = torch.Generator().manual_seed(42)
hard_sampler = InterventionSampler(N=N_demo, T=T_demo, p_hard=1.0, p_soft=0.0, p_time_varying=0.0, generator=gen)
hard_intv = hard_sampler.sample()

ax = axes[0, 0]
t_range = np.arange(T_demo)
signal = np.zeros(T_demo)
for t in hard_intv.times:
    signal[t] = hard_intv.values
ax.fill_between(t_range, 0, signal, alpha=0.3, color='red')
ax.axhline(hard_intv.values, color='red', linestyle='--', alpha=0.5)
ax.set_title(f'HARD: do(X := {hard_intv.values:.2f})')
ax.set_xlabel('Time')
ax.set_ylabel('Intervention value')
ax.axvspan(hard_intv.times[0], hard_intv.times[-1], alpha=0.1, color='red')

# --- SOFT intervention ---
gen = torch.Generator().manual_seed(42)
soft_sampler = InterventionSampler(N=N_demo, T=T_demo, p_hard=0.0, p_soft=1.0, p_time_varying=0.0, generator=gen)
soft_intv = soft_sampler.sample()

ax = axes[0, 1]
signal = np.zeros(T_demo)
for t in soft_intv.times:
    signal[t] = soft_intv.values
ax.fill_between(t_range, 0, signal, alpha=0.3, color='orange')
ax.axhline(soft_intv.values, color='orange', linestyle='--', alpha=0.5)
ax.set_title(f'SOFT: f(Pa) + eps + {soft_intv.values:.2f}')
ax.set_xlabel('Time')
ax.set_ylabel('Additive shift')
ax.axvspan(soft_intv.times[0], soft_intv.times[-1], alpha=0.1, color='orange')

# --- TIME-VARYING interventions (4 profiles) ---
profile_names = ['Step', 'Ramp', 'Sinusoidal', 'Trajectory']
colors = ['green', 'purple', 'blue', 'brown']

for profile_idx in range(4):
    gen = torch.Generator().manual_seed(profile_idx * 100 + 7)
    tv_sampler = InterventionSampler(N=N_demo, T=T_demo, p_hard=0.0, p_soft=0.0, p_time_varying=1.0, generator=gen)
    tv_intv = tv_sampler.sample()
    
    if profile_idx < 2:
        ax = axes[0, 2] if profile_idx == 0 else axes[1, 0]
    else:
        ax = axes[1, profile_idx - 1]
    
    signal = np.zeros(T_demo)
    for t in tv_intv.times:
        signal[t] = tv_intv.values(t)
    ax.plot(t_range, signal, color=colors[profile_idx], linewidth=1.5)
    ax.fill_between(t_range, 0, signal, alpha=0.2, color=colors[profile_idx])
    ax.set_title(f'TIME-VARYING: {profile_names[profile_idx]}')
    ax.set_xlabel('Time')
    ax.set_ylabel('c(t)')
    ax.axvspan(tv_intv.times[0], tv_intv.times[-1], alpha=0.1, color=colors[profile_idx])

plt.suptitle('Intervention Types in CausalTimePrior', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Hard intervention: targets={hard_intv.targets}, time=[{hard_intv.times[0]},{hard_intv.times[-1]}], value={hard_intv.values:.3f}")
print(f"Soft intervention: targets={soft_intv.targets}, time=[{soft_intv.times[0]},{soft_intv.times[-1]}], shift={soft_intv.values:.3f}")

Hard intervention: targets=[1, 3], time=[14,84], value=-2.246
Soft intervention: targets=[1, 3], time=[14,84], shift=-1.123


/tmp/ipykernel_12977/2466984059.py:66: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 6. How Many Interventions Are Applied?

Per paired sample, **exactly one `InterventionSpec`** is generated (`prior.py:205-210`):

- **Number of targets**: 1–2 variables (sampled uniformly, `max_targets=2`)
- **Time window**: contiguous period of length ∈ [10, T-10]
- **Start time**: sampled to ensure ≥ 10 steps before and after the intervention
- All targets share the **same type, timing, and values**

This means each sample has one intervention event affecting 1–2 variables simultaneously over one contiguous time period.

In [13]:
# Demonstrate intervention statistics over many samples
gen = torch.Generator().manual_seed(42)
N_demo, T_demo = 5, 100

n_samples = 500
types = []
num_targets = []
durations = []
start_times = []

for _ in range(n_samples):
    sampler = InterventionSampler(N=N_demo, T=T_demo, generator=gen)
    intv = sampler.sample()
    types.append(intv.intervention_type.value)
    num_targets.append(len(intv.targets))
    durations.append(len(intv.times))
    start_times.append(intv.times[0])

fig, axes = plt.subplots(1, 4, figsize=(16, 3))

# Type distribution
type_counts = {t: types.count(t) for t in ['hard', 'soft', 'time_varying']}
axes[0].bar(type_counts.keys(), type_counts.values(), color=['#e74c3c', '#f39c12', '#3498db'])
axes[0].set_title('Intervention type')
axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(type_counts.items()):
    axes[0].text(i, v + 5, f'{v/n_samples:.0%}', ha='center', fontsize=9)

# Number of targets
axes[1].hist(num_targets, bins=[0.5, 1.5, 2.5], rwidth=0.7, color='steelblue')
axes[1].set_title('Number of targets')
axes[1].set_xlabel('# targets')
axes[1].set_xticks([1, 2])

# Duration
axes[2].hist(durations, bins=30, color='coral', alpha=0.7)
axes[2].set_title('Intervention duration')
axes[2].set_xlabel('Length (timesteps)')
axes[2].axvline(np.mean(durations), color='red', linestyle='--', label=f'mean={np.mean(durations):.0f}')
axes[2].legend(fontsize=8)

# Start time
axes[3].hist(start_times, bins=30, color='mediumpurple', alpha=0.7)
axes[3].set_title('Intervention start time')
axes[3].set_xlabel('Start timestep')

plt.suptitle(f'Intervention statistics ({n_samples} samples, N={N_demo}, T={T_demo})', fontsize=11)
plt.tight_layout()
plt.show()

print(f"1 target: {num_targets.count(1)/n_samples:.0%}, 2 targets: {num_targets.count(2)/n_samples:.0%}")
print(f"Duration: mean={np.mean(durations):.0f}, min={min(durations)}, max={max(durations)}")

1 target: 49%, 2 targets: 51%
Duration: mean=50, min=10, max=90


/tmp/ipykernel_12977/3039462650.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 7. Regime-Switching SCMs

15% of sampled SCMs are **regime-switching** (`prior.py:126`). These model structural breaks where the causal structure changes over time.

### How it works

1. **Number of regimes**: $R \sim \text{Uniform}\{2, 3\}$
2. **Each regime** has its own:
   - Temporal DAG (G_0 + G_lags) — different graph structure
   - Mechanism weights and activations — different functional relationships
3. **Shared** across regimes: noise distributions
4. **Transition matrix**: each row sampled from a **sticky Dirichlet** prior

### Transition matrix

$$P_{ij} \sim \text{Dirichlet}(\alpha), \quad \alpha_i = 9.0, \quad \alpha_{j \neq i} = 0.5$$

The high self-transition concentration ($\alpha_{ii} = 9.0$) means regimes are **sticky** — the system tends to stay in the current regime for many timesteps before switching.

Expected self-transition probability: $\mathbb{E}[P_{ii}] = \alpha_i / \sum_j \alpha_j$
- R=2: $\alpha = [9.0, 0.5]$, so $\mathbb{E}[P_{ii}] = 9/9.5 \approx 94.7\%$
- R=3: $\alpha = [9.0, 0.5, 0.5]$, so $\mathbb{E}[P_{ii}] = 9/10 = 90\%$

This gives an expected regime duration of ~10–19 timesteps (geometric distribution).

In [14]:
# Demonstrate regime-switching: sample transition matrix
gen = torch.Generator().manual_seed(42)

for num_regimes in [2, 3]:
    print(f"\n=== {num_regimes} Regimes ===")
    
    # Sample transition matrix (same code as regime_switching_builder.py:74-83)
    P = np.zeros((num_regimes, num_regimes))
    for i in range(num_regimes):
        alpha = np.ones(num_regimes) * 0.5
        alpha[i] = 9.0
        np_seed = int(torch.randint(0, 2**31, (1,), generator=gen).item())
        rng = np.random.default_rng(np_seed)
        P[i] = rng.dirichlet(alpha)
    
    print("Transition matrix P:")
    for i in range(num_regimes):
        row_str = "  [" + ", ".join(f"{P[i,j]:.3f}" for j in range(num_regimes)) + "]"
        print(row_str)
    print(f"  Self-transition probs: {[f'{P[i,i]:.1%}' for i in range(num_regimes)]}")
    
    # Simulate regime sequence
    T_sim = 200
    regime_seq = np.zeros(T_sim, dtype=int)
    regime_seq[0] = 0
    for t in range(1, T_sim):
        regime_seq[t] = np.random.choice(num_regimes, p=P[regime_seq[t-1]])
    
    print(f"  Regime sequence (first 50): {regime_seq[:50].tolist()}")
    
    # Count regime durations
    durations = []
    current = regime_seq[0]
    count = 1
    for t in range(1, T_sim):
        if regime_seq[t] == current:
            count += 1
        else:
            durations.append(count)
            current = regime_seq[t]
            count = 1
    durations.append(count)
    print(f"  Regime durations: {durations}")
    print(f"  Mean duration: {np.mean(durations):.1f} timesteps")


=== 2 Regimes ===
Transition matrix P:
  [0.867, 0.133]
  [0.121, 0.879]
  Self-transition probs: ['86.7%', '87.9%']
  Regime sequence (first 50): [0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
  Regime durations: [2, 5, 5, 18, 4, 4, 6, 13, 13, 3, 16, 2, 14, 5, 3, 11, 11, 11, 5, 14, 14, 3, 1, 8, 1, 8]
  Mean duration: 7.7 timesteps

=== 3 Regimes ===
Transition matrix P:
  [0.974, 0.000, 0.026]
  [0.145, 0.840, 0.015]
  [0.139, 0.207, 0.653]
  Self-transition probs: ['97.4%', '84.0%', '65.3%']
  Regime sequence (first 50): [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
  Regime durations: [63, 1, 21, 22, 2, 4, 79, 4, 4]
  Mean duration: 22.2 timesteps


In [15]:
# Build and simulate a regime-switching SCM
gen = torch.Generator().manual_seed(55)
prior = CausalTimePrior(seed=55)

N_rs, K_rs = 4, 2
rs_builder = RegimeSwitchingSCMBuilder(
    num_nodes=N_rs,
    max_lag=K_rs,
    activations=prior.activations,
    gamma=0.7,
    sigma_w=1.0,
    sigma_b=0.5,
)
rs_scm = rs_builder.sample(gen)

print(f"Number of regimes: {rs_scm.num_regimes}")
print(f"Transition matrix:")
for i in range(rs_scm.num_regimes):
    print(f"  {rs_scm.transition_matrix[i].round(3)}")

# Show what differs between regimes
for r in range(rs_scm.num_regimes):
    dag = rs_scm.dags[r]
    print(f"\nRegime {r}:")
    print(f"  G_0 edges: {list(dag.G_0.edges())}")
    for k in range(K_rs):
        nnz = int(dag.G_lags[k].sum())
        print(f"  G_{k+1} edges: {nnz}")

Number of regimes: 3
Transition matrix:
  [0.841 0.099 0.06 ]
  [0.043 0.866 0.091]
  [0.    0.006 0.994]

Regime 0:
  G_0 edges: [('X0', 'X2'), ('X2', 'X3')]
  G_1 edges: 3
  G_2 edges: 4

Regime 1:
  G_0 edges: [('X0', 'X2'), ('X0', 'X3'), ('X1', 'X2'), ('X1', 'X3'), ('X2', 'X3')]
  G_1 edges: 8
  G_2 edges: 6

Regime 2:
  G_0 edges: [('X0', 'X1'), ('X0', 'X2'), ('X1', 'X2'), ('X2', 'X3')]
  G_1 edges: 2
  G_2 edges: 4


In [16]:
# Generate regime-switching observational data and visualize
X_rs, regimes = rs_scm.sample_observational(T=200, burn_in=50, return_regimes=True)

fig, axes = plt.subplots(2, 1, figsize=(14, 5), gridspec_kw={'height_ratios': [1, 3]})

# Regime indicator
ax = axes[0]
colors = ['#e74c3c', '#3498db', '#2ecc71']
for r in range(rs_scm.num_regimes):
    mask = regimes == r
    ax.fill_between(range(len(regimes)), 0, 1, where=mask, 
                    color=colors[r], alpha=0.5, label=f'Regime {r}')
ax.set_ylabel('Regime')
ax.set_yticks([])
ax.legend(loc='upper right', fontsize=8, ncol=rs_scm.num_regimes)
ax.set_xlim(0, len(regimes))
ax.set_title('Regime-Switching SCM: Observational Data')

# Time series
ax = axes[1]
for i in range(X_rs.shape[1]):
    ax.plot(X_rs[:, i].numpy(), label=f'X{i}', alpha=0.8, linewidth=0.8)
ax.set_xlabel('Time')
ax.set_ylabel('Value')
ax.legend(loc='upper right', fontsize=8, ncol=X_rs.shape[1])
ax.set_xlim(0, X_rs.shape[0])

plt.tight_layout()
plt.show()

print(f"Time series shape: {X_rs.shape}")
print(f"Regime sequence (unique): {np.unique(regimes)}")

Time series shape: torch.Size([200, 4])
Regime sequence (unique): [1 2]


/tmp/ipykernel_12977/1431297888.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 8. Full Example: Generate and Visualize a Paired Sample

Putting it all together — generate a paired (observational, interventional) sample and visualize the intervention effect.

In [17]:
# Generate a paired example
prior = CausalTimePrior(seed=100)
X_obs, X_int, intervention, scm = prior.generate_pair(T=100)

N = X_obs.shape[1]
T = X_obs.shape[0]
target = intervention.targets[0]
t_start = intervention.times[0]
t_end = intervention.times[-1]

print(f"SCM: N={N} variables, K={scm.dag.K} lags")
print(f"Graph G_0 edges: {list(scm.dag.G_0.edges())}")
print(f"Intervention: {intervention.intervention_type.value} on target(s) {intervention.targets}")
print(f"  Active time: [{t_start}, {t_end}] ({t_end - t_start + 1} steps)")
if isinstance(intervention.values, (int, float)):
    print(f"  Value: {intervention.values:.3f}")
else:
    print(f"  Value: time-varying function")

# Plot observational vs interventional for all variables
fig, axes = plt.subplots(N, 1, figsize=(14, 2.5 * N), sharex=True)
if N == 1:
    axes = [axes]

for i in range(N):
    ax = axes[i]
    ax.plot(X_obs[:, i].numpy(), 'b-', alpha=0.7, linewidth=1, label='Observational')
    ax.plot(X_int[:, i].numpy(), 'r-', alpha=0.7, linewidth=1, label='Interventional')
    ax.axvspan(t_start, t_end, alpha=0.15, color='red')
    ax.set_ylabel(f'X{i}')
    if i in intervention.targets:
        ax.set_ylabel(f'X{i} (target)', fontweight='bold', color='red')
    if i == 0:
        ax.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Time')
plt.suptitle(f'Paired Sample: {intervention.intervention_type.value} intervention on X{intervention.targets}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

SCM: N=5 variables, K=2 lags
Graph G_0 edges: [('X0', 'X1'), ('X1', 'X2'), ('X2', 'X3'), ('X3', 'X4')]
Intervention: hard on target(s) [0]
  Active time: [38, 63] (26 steps)
  Value: 1.870


/tmp/ipykernel_12977/812086518.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Show intervention effect: difference between obs and int
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Intervention effect on target
ax = axes[0]
diff = (X_int[:, target] - X_obs[:, target]).numpy()
ax.plot(diff, 'k-', linewidth=1)
ax.fill_between(range(T), 0, diff, where=np.abs(diff) > 0, alpha=0.3, color='red')
ax.axvspan(t_start, t_end, alpha=0.1, color='red')
ax.set_title(f'Intervention effect on X{target} (target)')
ax.set_xlabel('Time')
ax.set_ylabel('X_int - X_obs')
ax.axhline(0, color='gray', linewidth=0.5)

# Downstream effects on non-target variables
ax = axes[1]
non_targets = [i for i in range(N) if i not in intervention.targets]
for i in non_targets:
    diff_i = (X_int[:, i] - X_obs[:, i]).numpy()
    ax.plot(diff_i, linewidth=1, alpha=0.7, label=f'X{i}')
ax.axvspan(t_start, t_end, alpha=0.1, color='red')
ax.set_title('Downstream causal effects on non-target variables')
ax.set_xlabel('Time')
ax.set_ylabel('X_int - X_obs')
ax.axhline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

/tmp/ipykernel_12977/3862143868.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 9. Design Choices: Identifiability, Noise, and Effect Propagation

This section addresses common questions about CausalTimePrior's modeling assumptions.

### Lag complexity is linear, not exponential

Each additional lag $k$ adds one $N \times N$ Bernoulli adjacency matrix $G_k$ and one scalar weight per (node, parent) pair. Total weights per node scale as $O(N \times (1 + K))$—**linear in K**. Edge density further decreases with lag via geometric decay:

| Lag | Edge prob ($p=0.3$) | Expected edges ($N=5$) |
|-----|---------------------|----------------------|
| $G_0$ | 0.300 | ~3 (DAG, max $N(N-1)/2$) |
| $G_1$ | 0.210 | ~5.3 (full $N^2$ matrix) |
| $G_2$ | 0.147 | ~3.7 |
| $G_3$ | 0.103 | ~2.6 |

### Per-timestep identifiability

At each time step $t$, variables are evaluated in **topological order** of $G_0$. When computing $X_i(t)$:
- **Instantaneous parents** $X_j(t)$ are already computed (topological order guarantees this)
- **Lagged parents** $X_j(t-k)$ are fully determined from prior time steps

Lagged edges never break identifiability—they reference already-fixed past values. The additive noise model (ANM) structure $\sigma(\sum w_j \cdot x_j + b) + \epsilon$ combined with non-Gaussian noise (Uniform, Laplace) further supports identifiability under standard ANM results.

### Instantaneous vs. temporal causality

The model handles both through a **unified forward simulation**, not separate modules:
- **Instantaneous effects** ($G_0$): same-timestep dependencies, resolved via topological ordering. Analogous to **imputation**—filling in values given other current-time observations.
- **Temporal effects** ($G_1, \ldots, G_K$): cross-timestep dependencies, acyclic by construction (time flows forward). Analogous to **forecasting**—predicting current values from past observations.

$G_0$ requires a DAG constraint (acyclicity), while $G_k$ matrices can have arbitrary structure including feedback loops (e.g., $X_1(t-1) \to X_2(t)$ and $X_2(t-1) \to X_1(t)$).

In [ ]:
# Demonstrate effect propagation and decay after intervention
prior = CausalTimePrior(seed=200)
T_demo = 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel 1: Effect decay across multiple SCMs
ax = axes[0]
for seed in [200, 201, 202, 203, 204]:
    prior_i = CausalTimePrior(seed=seed)
    X_obs, X_int, intv, scm = prior_i.generate_pair(T=T_demo)
    
    # Compute mean absolute effect across all variables
    effect = (X_int - X_obs).abs().mean(dim=1).numpy()
    t_end = intv.times[-1]
    
    # Normalize time relative to intervention end
    t_rel = np.arange(T_demo) - t_end
    ax.plot(t_rel, effect, alpha=0.5, linewidth=1, label=f'SCM {seed}')

ax.axvline(0, color='red', linestyle='--', alpha=0.5, label='Intervention ends')
ax.set_xlabel('Time relative to intervention end')
ax.set_ylabel('Mean |effect| across variables')
ax.set_title('Causal effect decay after intervention')
ax.legend(fontsize=7, loc='upper right')
ax.set_xlim(-30, 30)

# Panel 2: PFN query horizon distribution
ax = axes[1]
# Simulate the query time distribution from generate_dataset_v2.py
n_queries = 5000
downstream_prob = 0.7
query_horizons = []
for _ in range(n_queries):
    is_downstream = np.random.rand() < downstream_prob
    if is_downstream:
        horizon = np.random.randint(1, 6)  # 1-5 steps
    else:
        horizon = 0  # instantaneous
    query_horizons.append(horizon)

ax.hist(query_horizons, bins=np.arange(-0.5, 6.5, 1), rwidth=0.7,
        color='steelblue', edgecolor='white')
ax.set_xlabel('Query horizon (steps after intervention)')
ax.set_ylabel('Count')
ax.set_title(f'PFN training query distribution\n(70% downstream @ 1-5 steps, 30% instantaneous)')
ax.set_xticks(range(6))
ax.set_xticklabels(['0\n(instant)', '1', '2', '3', '4', '5'])

for i in range(6):
    count = query_horizons.count(i)
    ax.text(i, count + 50, f'{count/n_queries:.0%}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print(f"Query horizon distribution: {dict(sorted([(h, query_horizons.count(h)) for h in set(query_horizons)]))}")
print(f"Mean query horizon: {np.mean(query_horizons):.2f} steps")

### Noise assumptions

**Additive and exogenous:** Noise $\epsilon_t^{(i)}$ is added *after* the nonlinear activation and sampled independently of parent values:
$$X_t^{(i)} = \sigma\left(\sum_j w_j \cdot \text{Pa}_j + b\right) + \epsilon_t^{(i)}$$

**Markovian:** Noise is i.i.d. across time — each timestep samples fresh noise with no temporal correlation. This means the causal graph $\mathcal{G}$ fully mediates **all** temporal dependencies. There are no hidden common causes acting across time beyond what the lagged edges encode.

**Non-Markovian confounding (future extension):** If noise were correlated across time — $\text{Cov}(\epsilon_t^{(i)}, \epsilon_{t-s}^{(j)}) \neq 0$ — this would model latent confounders that persist over time (e.g., unobserved economic trends, hidden regime states). This would break the assumption that the causal graph captures all temporal dependencies, making interventional distributions depend on unobserved confounder trajectories.

### Effect propagation and the Markovian property

Causal effects decay through three mechanisms:
1. **Maximum lag $K \in \{1,2,3\}$:** No direct causal influence beyond $K$ steps
2. **Geometric edge decay ($\gamma = 0.7$):** Distant lags have fewer edges
3. **Bounded activations** (tanh, sin, etc.): Saturating nonlinearities dampen signal propagation

There is **no explicit spectral radius constraint** on the weight matrices. Stability is enforced empirically via value clipping ([-1000, 1000]) and divergence rejection. The left panel above shows that effects typically decay within a few steps after the intervention window ends.

This rapid decay, combined with Markovian noise, means the model assumes all relevant causal information is contained in the last $K = 1$–$3$ time steps. **Identifiability holds within this lag window** under the ANM structure, but if real-world causal effects persist longer than $K$ steps, they would appear as unmodeled confounders.

### Stationarity scope

| SCM type | Fraction | Stationarity |
|----------|----------|-------------|
| Diverse nonlinear | 70% | Stationary (fixed coefficients, graph, noise) |
| Chain | 15% | Stationary |
| Regime-switching | 15% | Piecewise-stationary (2–3 regimes, Markov transitions) |

**Not yet supported:** continuously time-varying coefficients, one-time structural breaks, heteroscedastic (time-varying) noise variance. These are natural extensions for non-stationary settings.

---
## Summary

| Component | Prior | Default |
|-----------|-------|---------|
| **N** (variables) | Uniform[3, 10] | — |
| **K** (lags) | Uniform{1, 2, 3} | — |
| **Edge prob** p | Beta(2, 5) | mean ≈ 0.286 |
| **Lag decay** | p_k = p · 0.7^k | gamma = 0.7 |
| **Activation** | Uniform over 9 functions | — |
| **Weights** | N(0, sigma_w^2) | sigma_w = 1.0 |
| **Bias** | N(0, sigma_b^2) | sigma_b = 0.5 |
| **Root noise** | N(0, std), std ~ ShiftedExp(1.0, 0.1) | — |
| **Non-root noise** | 1/3 Gaussian, 1/3 Uniform, 1/3 Laplace | std ~ ShiftedExp(10.0, 0.01) |
| **Intervention type** | 50% hard, 30% soft, 20% time-varying | — |
| **Targets** | 1–2 variables | max_targets = 2 |
| **Duration** | Uniform[10, T-10] | contiguous window |
| **SCM type** | 70% diverse, 15% chain, 15% regime-switching | — |
| **Regimes** | Uniform{2, 3}, Dirichlet(sticky) | ~90% self-transition |
| **Burn-in** | 50 steps (discarded) | — |
| **Clipping** | [-1000, 1000] per step | — |
| **Divergence** | Reject if |x| > 10^6 or NaN/Inf | — |